# Goal-prediction labels and attribution audit

The prediction target is whether the same team scores a goal before its current
possession ends, given only the actions observed so far.

This notebook inspects labels for **2023–2025**, using the existing inferred
possessions and pre-shot prefixes. It does not fit a model, select a model or
load the 2026 test season. The earlier shot experiments and their saved outputs
remain available in notebooks 03–05.

Prepare the three development seasons using the README, then run
`python -m afl_opposition.goal_labels` from the repository root. That command
verifies the sources, constructs labels and saves an audit. Rerunning it requires
`--overwrite`; rerunning this notebook only reads those saved results.

Run cells in order using the **Python (afl-opposition-analysis)** kernel. Saved
outputs document the label audit.
See [the label specification](../docs/goal-labels.md) for the complete rules.

## 1. Saved labels and provenance verification

The loader resolves paths relative to the repository and verifies SHA-256 hashes
for the label tables and the source modules used to generate them. A changed
artifact or implementation requires a regenerated audit to keep the report
consistent with its recorded provenance.

`examples` contains model inputs, targets and weights. `audit` has one row per
possession. `goals` has one row per raw goal annotation, including quarantined
quarters. `coverage` summarises seasons. This stage performs no model fitting.

In [1]:
import hashlib
import json
from pathlib import Path

import pandas as pd
from IPython.display import display

from afl_opposition.goal_labels import GOAL_LABEL_VERSION
from afl_opposition.preprocess import FEATURE_COLUMNS

project_root = next(
    (p for p in [Path.cwd(), *Path.cwd().parents]
     if (p / "data/sources.json").is_file() and (p / "pyproject.toml").is_file()), None,
)
if project_root is None:
    raise RuntimeError("Launch from the repository root or notebooks folder.")
directory = project_root / "artifacts/goal-labels-2023-2025"
if not (directory / "summary.json").is_file():
    raise RuntimeError("Prepare 2023–2025, then run python -m afl_opposition.goal_labels.")
summary = json.loads((directory / "summary.json").read_text())

def file_hash(path):
    with path.open("rb") as stream:
        return hashlib.file_digest(stream, "sha256").hexdigest()

for name, expected in summary["artifact_sha256"].items():
    if file_hash(directory / f"{name}.parquet") != expected:
        raise RuntimeError(f"Changed artifact: {name}. Regenerate the goal-label audit.")
for name, expected in summary["code_sha256"].items():
    if file_hash(project_root / "src/afl_opposition" / name) != expected:
        raise RuntimeError(f"Changed implementation: {name}. Regenerate the goal-label audit.")
assert summary["goal_label_version"] == GOAL_LABEL_VERSION
assert summary["seasons"] == [2023, 2024, 2025]
examples = pd.read_parquet(directory / "examples.parquet")
audit = pd.read_parquet(directory / "possession_audit.parquet")
goals = pd.read_parquet(directory / "goal_annotation_audit.parquet")
coverage = pd.read_parquet(directory / "coverage.parquet")
eligible = audit.loc[audit.status.eq("eligible")].copy()
print("Verified artifacts:", directory.relative_to(project_root).as_posix())
print("Goal label version:", GOAL_LABEL_VERSION)
print("Feature columns:", FEATURE_COLUMNS)

Verified artifacts: artifacts/goal-labels-2023-2025
Goal label version: 0.1.0
Feature columns: ['action_type', 'x_norm', 'y_norm', 'gap_seconds', 'elapsed_seconds']


## 2. Prediction target and input cutoff

`target = 1` means the possessing team scores before the inferred episode ends.
`target = 0` means the episode has a supported ending without such a goal.
Ambiguous or incomplete episodes have no label and contribute no examples.

The **input cutoff stays before the first explicit shot**. If there is no shot,
every retained action in an eligible possession supplies a prefix. In a sequence
`gather → handball → shot → goal`, the first two actions each create an example;
the shot and goal do not. The eventual goal is the training target, not a feature.

The table counts possessions, not rows. The combination `shot target = 0, goal
target = 1` should be absent because the existing eligibility policy excludes
scores without an explicit shot flag. Goal prediction remains distinct from
predicting whether a shot is converted once that shot has already been taken.

In [2]:
display(pd.crosstab(
    eligible.shot_target, eligible.target,
    rownames=["Shot target"], colnames=["Goal target"], dropna=False,
).reindex(index=[0, 1], columns=[0, 1], fill_value=0))
print("Target:", summary["target_definition"])
print("Input cutoff:", summary["input_cutoff"])

Goal target,0,1
Shot target,,
0,123105,0
1,15557,15160


Target: The same team scores a recorded goal before the inferred possession ends.
Input cutoff: Strictly before the first explicit shot; all retained actions if there is no shot.


## 3. Label prevalence by season

The development split assigns 2023–2024 to training and 2025 to validation. Multiple prefixes
come from each possession, so the number of example rows is larger than the
number of possessions. `goal_possession_rate` is the positive fraction when each
possession receives equal weight; it is also the prevalence reference for
possession-weighted average precision on this target.

These counts describe label coverage and prevalence. Model performance is
evaluated separately in the goal-modelling notebooks.

In [3]:
display(coverage[[
    "season", "matches", "eligible_possessions", "goal_possessions",
    "no_goal_possessions", "prefix_examples", "goal_possession_rate",
]].style.format({"goal_possession_rate": "{:.2%}"}))
print("Eligible possessions:", len(eligible))
print("Labelled prefixes:", len(examples))

,season,matches,eligible_possessions,goal_possessions,no_goal_possessions,prefix_examples,goal_possession_rate
0,2023,216,52429,5018,47411,285771,9.57%
1,2024,215,50649,5046,45603,277528,9.96%
2,2025,216,50744,5096,45648,281476,10.04%


Eligible possessions: 153822
Labelled prefixes: 844775


## 4. Goal attribution and exclusion accounting

Goal attribution finds the preceding retained action in the **same match and
quarter**, using `display_order`, not timestamps alone. Verification requires one
goal annotation, a `Goal` possession endpoint, matching closing time, and the
same known team. A wrong/missing team, duplicate goal, inconsistent endpoint or
missing annotation excludes an otherwise eligible episode.

Quarantined quarters stay in the goal-annotation audit but never enter the
examples. A verified goal can still be excluded by the earlier shot policy:
unsupported events, unresolved endings, missing shot flags, shot-marker conflicts
or a shot as the first action. The tables reconcile these stages separately;
quarantined quarters were removed before possession construction.

In [4]:
display(coverage[[
    "season", "raw_goals", "quarantined_goals", "verified_goals", "goal_possessions",
]])
display(goals.groupby(["season", "status", "possession_status"], dropna=False)
        .size().rename("goal_annotations").reset_index())
display(audit.groupby(["season", "status"]).size().unstack(fill_value=0))
print("Additional exclusions from goal attribution:", summary["additional_goal_exclusions"])

,season,raw_goals,quarantined_goals,verified_goals,goal_possessions
0,2023,5230,0,5230,5018
1,2024,5258,8,5250,5046
2,2025,5297,11,5286,5096


,season,status,possession_status,goal_annotations
0,2023.0,verified,eligible,5018
1,2023.0,verified,no_pre_shot_action,11
2,2023.0,verified,preprocessing_review,200
3,2023.0,verified,score_without_shot_flag,1
4,2024.0,quarantined_quarter,NaN,8
5,2024.0,verified,eligible,5046
6,2024.0,verified,no_pre_shot_action,10
7,2024.0,verified,preprocessing_review,192
8,2024.0,verified,score_without_shot_flag,2
9,2025.0,quarantined_quarter,NaN,11


status,eligible,no_pre_shot_action,preprocessing_review,score_without_shot_flag,shot_marker_conflict
season,,,,,
2023,52429,32,3698,1,0
2024,50649,28,3369,2,1
2025,50744,42,3245,2,2


Additional exclusions from goal attribution: 0


## 5. A possession that ends in a goal

The helper below reads a season's verified source tables and selects a complete
passage by possession ID. It shows raw annotations from the first retained action
through the closing event. For a turnover, that includes the opponent's first
controlled action to explain why the possession ended.

`included_as_input` identifies the retained pre-shot actions that supply training
examples. Auxiliary annotations, shots, post-shot actions and outcome rows are
displayed for auditing but are excluded from the model inputs. `goal_target` is
shown only on actual examples. All other rows display a missing target.
`handling` explains source rows such as `boundary_duplicate`: these repeats stay
visible in the audit but do not create additional model actions.

The selection is deterministic and favours a short passage for readability.
The second table is the actual feature matrix with its labels and sample weights.

In [5]:
season_cache = {}

def load_season_for_display(season):
    if season not in season_cache:
        tables = {}
        for name in ["events", "annotations"]:
            path = project_root / f"data/processed/{season}/{name}.parquet"
            expected = summary["sources"][str(season)]["processed_file_sha256"][f"{name}.parquet"]
            if file_hash(path) != expected:
                raise RuntimeError("Preprocessed data changed; regenerate the goal-label audit.")
            tables[name] = pd.read_parquet(path)
        season_cache[season] = tables
    return season_cache[season]

def choose_possession(candidates):
    if candidates.empty:
        raise ValueError("No eligible passage matches this example.")
    return candidates.sort_values(["event_count", "season", "match_id", "period", "start_seconds"]).iloc[0]

def show_possession(record):
    tables = load_season_for_display(int(record.season))
    actions = tables["events"].loc[tables["events"].possession_id.eq(record.possession_id)]
    quarter = tables["annotations"].loc[
        tables["annotations"].match_id.eq(record.match_id)
        & tables["annotations"].period.eq(record.period)
    ].sort_values("display_order")
    start_order = int(actions.display_order.min())
    last_order = int(actions.display_order.max())
    if record.end_reason == "opponent_control":
        next_actions = tables["events"].loc[
            tables["events"].match_id.eq(record.match_id)
            & tables["events"].period.eq(record.period)
            & tables["events"].display_order.gt(last_order)
        ]
        end_order = int(next_actions.display_order.min())
    else:
        ending = quarter.loc[quarter.display_order.gt(last_order)
                             & quarter.description.eq(record.end_reason)
                             & quarter.period_seconds.eq(record.closed_at_seconds)]
        if ending.empty:
            raise ValueError("This display helper requires an explicit closing event.")
        end_order = int(ending.display_order.min())
    passage = quarter.loc[quarter.display_order.between(start_order, end_order)].copy()
    inputs = examples.loc[examples.possession_id.eq(record.possession_id)].set_index("event_id")
    passage["included_as_input"] = passage.raw_event_id.isin(inputs.index)
    passage["goal_target"] = passage.raw_event_id.map(inputs.target).astype("Int64")
    print("Possession:", record.possession_id)
    print("Team:", record.team_id, "| endpoint:", record.end_reason,
          "| shot target:", int(record.shot_target), "| goal target:", int(record.target))
    display(passage[["display_order", "period_seconds", "description", "team_id", "handling",
                     "shot_at_goal", "included_as_input", "goal_target"]].reset_index(drop=True))
    return inputs.reset_index()

goal_example = choose_possession(eligible.loc[
    eligible.target.eq(1) & eligible.explicit_shot_count.eq(1)
    & eligible.eligible_prefix_count.ge(2) & eligible.event_count.le(8)
])
goal_inputs = show_possession(goal_example)
display(goal_inputs[["event_index", *FEATURE_COLUMNS, "target", "sample_weight"]])

Possession: CD_M20230140103:q3:c137:e1163:pos
Team: CD_T100 | endpoint: Goal | shot target: 1 | goal target: 1


,display_order,period_seconds,description,team_id,handling,shot_at_goal,included_as_input,goal_target
0,1163,382,Loose Ball Get,CD_T100,event,<NA>,True,1
1,1164,382,Loose Ball Get,CD_T100,boundary_duplicate,<NA>,False,<NA>
2,1165,383,Kick,CD_T100,event,<NA>,True,1
3,1166,383,Kick Into F50,CD_T100,paired_kick_annotation,<NA>,False,<NA>
4,1167,383,Kick Inside 50 Result,CD_T100,auxiliary_annotation,<NA>,False,<NA>
5,1168,386,Ground Kick,CD_T100,event,True,False,<NA>
6,1169,388,Goal,CD_T100,stoppage,<NA>,False,<NA>


,event_index,action_type,x_norm,y_norm,gap_seconds,elapsed_seconds,target,sample_weight
0,0,loose_ball_get,0.2125,0.062016,0,0,1,0.5
1,1,kick,0.275,0.093023,1,1,1,0.5


## 6. A shot followed by a behind

This passage has a shot target of one but a goal target of zero. A behind is a
different recorded scoring outcome from a goal; it does not satisfy the binary
goal target. The shot and its result still remain outside the input sequence.
A goal-negative label records the realised outcome. It does not imply that
the observed pre-shot state had zero probability of producing a goal.

In [6]:
behind_example = choose_possession(eligible.loc[
    eligible.end_reason.eq("Behind") & eligible.target.eq(0)
    & eligible.eligible_prefix_count.ge(2) & eligible.event_count.le(8)
])
behind_inputs = show_possession(behind_example)

Possession: CD_M20230140103:q2:c78:e700:pos
Team: CD_T100 | endpoint: Behind | shot target: 1 | goal target: 0


,display_order,period_seconds,description,team_id,handling,shot_at_goal,included_as_input,goal_target
0,700,581,Free For,CD_T100,event,<NA>,True,0
1,701,581,Free Advantage,CD_T100,event,<NA>,True,0
2,702,584,Kick,CD_T100,event,True,False,<NA>
3,703,584,Kick Into F50,CD_T100,paired_kick_annotation,<NA>,False,<NA>
4,704,584,Kick Inside 50 Result,<NA>,auxiliary_annotation,<NA>,False,<NA>
5,705,585,Behind,CD_T100,stoppage,<NA>,False,<NA>


## 7. A turnover without a shot

The opponent's controlled action closes the old team's possession and starts a
new one. It is shown as the boundary, not included as an input for the old team.
A goal by the opponent later in the match cannot turn this episode positive.
Both the shot and goal targets are zero for the possession displayed here.

In [7]:
turnover_example = choose_possession(eligible.loc[
    eligible.end_reason.eq("opponent_control") & eligible.shot_target.eq(0)
    & eligible.eligible_prefix_count.ge(2) & eligible.event_count.le(8)
])
turnover_inputs = show_possession(turnover_example)

Possession: CD_M20230140101:q1:c4:e34:pos
Team: CD_T120 | endpoint: opponent_control | shot target: 0 | goal target: 0


,display_order,period_seconds,description,team_id,handling,shot_at_goal,included_as_input,goal_target
0,34,169,Loose Ball Get Crumb,CD_T120,event,<NA>,True,0
1,35,169,Loose Ball Get Crumb,CD_T120,boundary_duplicate,<NA>,False,<NA>
2,36,171,Kick,CD_T120,event,<NA>,True,0
3,37,175,Gather from Opposition,CD_T30,event,<NA>,False,<NA>


## 8. A recovered miss followed by a goal

The target covers the entire possession. An explicit shot does not itself end an
episode: if play continues and the same team recovers the ball without a modelled
boundary, a later goal still makes the earlier prefixes positive. A recorded
behind or stoppage would end the episode instead.

For comparability with the existing dataset, inputs still stop before the **first**
shot. The recovery and second shot are evidence for the eventual label, not extra
input rows. This model therefore predicts from pre-shot states only; evaluating
post-shot states would require a separately specified dataset.

For clarity, the example selects a 2023 passage with a recorded gather, receive
or mark between two shots. The season summary counts all eligible multi-shot goal
possessions, including those without a separate recovery annotation.

In [8]:
actions_2023 = load_season_for_display(2023)["events"]
shot_actions = actions_2023.loc[actions_2023.is_explicit_shot]
first_shot = shot_actions.groupby("possession_id").event_index.min()
last_shot = shot_actions.groupby("possession_id").event_index.max()
recovery_rows = actions_2023.loc[
    actions_2023.event_index.gt(actions_2023.possession_id.map(first_shot))
    & actions_2023.event_index.lt(actions_2023.possession_id.map(last_shot))
    & actions_2023.action_type.isin(["gather", "hard_ball_get", "loose_ball_get", "mark", "receive"])
]
recovery_example = choose_possession(eligible.loc[
    eligible.target.eq(1) & eligible.explicit_shot_count.gt(1)
    & eligible.event_count.le(12) & eligible.possession_id.isin(recovery_rows.possession_id)
])
recovery_inputs = show_possession(recovery_example)
display(coverage[["season", "multi_shot_goal_possessions"]])

Possession: CD_M20230140404:q3:c128:e1135:pos
Team: CD_T140 | endpoint: Goal | shot target: 1 | goal target: 1


,display_order,period_seconds,description,team_id,handling,shot_at_goal,included_as_input,goal_target
0,1135,85,Gather from Opposition,CD_T140,event,<NA>,True,1
1,1136,85,Gather from Opposition,CD_T140,boundary_duplicate,<NA>,False,<NA>
2,1137,90,Kick,CD_T140,event,True,False,<NA>
3,1138,90,Shot At Goal,CD_T140,auxiliary_annotation,<NA>,False,<NA>
4,1139,93,Contested Mark,CD_T140,event,<NA>,False,<NA>
5,1140,112,Kick,CD_T140,event,True,False,<NA>
6,1141,114,Goal,CD_T140,stoppage,<NA>,False,<NA>


,season,multi_shot_goal_possessions
0,2023,87
1,2024,71
2,2025,91


## 9. Feature eligibility and possession weights

Each eligible possession receives total sample weight one, divided evenly across
its prefixes. That keeps long passages from dominating simply because they
produce more training rows. `X` contains only current action type, normalised
position and timing; `y` and weights are separate.

For sequences, the same rows can later be grouped into ordered prefixes. This
audit does not fit an encoder, scaler or classifier. Excluded possessions retain
missing targets rather than being incorrectly converted to negatives.

In [9]:
X = examples[FEATURE_COLUMNS]
y = examples.target
sample_weight = examples.sample_weight

assert y.isin([0, 1]).all()
assert set(examples.possession_id) == set(eligible.possession_id)
assert X.notna().all().all()
assert audit.loc[audit.status.ne("eligible"), "target"].isna().all()
assert examples.groupby("possession_id").target.nunique().eq(1).all()
weight_totals = examples.groupby("possession_id").sample_weight.sum()
assert weight_totals.sub(1).abs().lt(1e-10).all()
cutoff = examples.possession_id.map(audit.set_index("possession_id").first_shot_index)
assert (cutoff.isna() | examples.event_index.lt(cutoff)).all()
assert not {"goal_seconds", "end_reason", "shot_target", "first_shot_index", "is_explicit_shot"}.intersection(X.columns)
print("X shape:", X.shape)
print("y shape:", y.shape)
print("Total sample weight:", round(float(sample_weight.sum())))
print("All label, cutoff and weighting checks passed.")

X shape: (844775, 5)
y shape: (844775,)
Total sample weight: 153822
All label, cutoff and weighting checks passed.


## 10. Modelling inputs and limitations

The saved tables support a goal-prediction baseline using the same five features.
The goal baseline fits on 2023–2024, selects regularisation using 2025, then
refits on the combined development data before test evaluation. This notebook
establishes label provenance and coverage; it reports no model performance. The 2026 season has already
been used for the shot benchmark; it is not a newly untouched holdout.

Goal attribution verifies consistency within the recorded data. The possession
boundaries remain project-specific approximations, and the 2024 source is missing
one fixture. Existing quarter and episode exclusions remain part of the coverage
limitations. Goals after a stoppage or opponent control fall outside this target.

In [10]:
display(pd.DataFrame({
    "artifact": ["examples.parquet", "possession_audit.parquet", "goal_annotation_audit.parquet", "coverage.parquet", "summary.json"],
    "purpose": ["Pre-shot features, goal target and weights", "Eligibility and shot/goal targets per possession",
                "Every raw goal and its attribution status", "Season counts and prevalence", "Rules, source checksums and artifact checksums"],
}))
print("Label audit complete. No goal model has been trained.")

,artifact,purpose
0,examples.parquet,"Pre-shot features, goal target and weights"
1,possession_audit.parquet,Eligibility and shot/goal targets per possession
2,goal_annotation_audit.parquet,Every raw goal and its attribution status
3,coverage.parquet,Season counts and prevalence
4,summary.json,"Rules, source checksums and artifact checksums"


Label audit complete. No goal model has been trained.
